# MoCo v2 Pretraining — Google Colab (L4)

**세션 재시작 시**: Cell 1~3(Setup)만 다시 실행하고 Cell 4(학습)는 자동으로 마지막 체크포인트부터 이어서 시작합니다.

**체크포인트 / 데이터 / 로그**는 Google Drive에 저장되어 세션이 끊겨도 유지됩니다.

In [ ]:
# Cell 1 — GPU 확인
import torch
assert torch.cuda.is_available(), 'GPU가 없습니다. 런타임 → 런타임 유형 변경 → L4 GPU 선택'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2 — Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — 레포 클론 / 업데이트 + 패키지 설치 + Drive 심링크
import os, subprocess, shutil

REPO     = 'Visual-Intelligence-Learning-STL-DINOFORCE-TV'
REPO_URL = 'https://github.com/hamseungyeal/Visual-Intelligence-Learning-STL-DINOFORCE-TV.git'
REPO_DIR = f'/content/{REPO}'
DRIVE_DIR = f'/content/drive/MyDrive/ssl_project'   # Drive 저장 경로

# Drive에 영구 보관 폴더 생성
for sub in ['data', 'outputs/mocov2_r50_seed42', 'logs']:
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)

# 레포 클론 or 최신화
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# 패키지 설치
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)

# data / outputs / logs → Drive 심링크 (세션 재시작 후에도 데이터 유지)
for d in ['data', 'outputs', 'logs']:
    local  = f'{REPO_DIR}/{d}'
    remote = f'{DRIVE_DIR}/{d}'
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        os.remove(local)
    elif os.path.exists(local):
        shutil.rmtree(local)
    os.symlink(remote, local)
    print(f'  linked: {local} → {remote}')

print('\nSetup 완료!')

In [ ]:
# Cell 4 — 학습 시작 (자동 resume)
import glob, os

# 마지막 체크포인트 자동 탐색
ckpts = sorted(glob.glob('outputs/mocov2_r50_seed42/ckpt_ep*.pth'))
if ckpts:
    print(f'Resume: {ckpts[-1]}')
    resume_arg = f'--resume {ckpts[-1]}'
else:
    print('처음부터 학습 시작')
    resume_arg = ''

# Colab: num_workers=2 (8이면 불안정), 나머지는 config 그대로
os.system(f'python3 scripts/train_mocov2.py --num-workers 2 {resume_arg}')

In [ ]:
# (선택) 로그 실시간 확인
# !tail -f logs/mocov2_seed42.log